# FleetLogix — Avance 3: Data Warehouse y ETL
**Proyecto integrador · Módulo 2 · Henry**

## 1. Introducción
Diseñar e implementar un Data Warehouse en Snowflake a partir de la base operacional PostgreSQL de FleetLogix, utilizando un modelo estrella y un pipeline ETL en Python.

## 2. Objetivos
- Implementar una tabla de hechos y seis dimensiones.
- Configurar Snowflake, escalabilidad, Time Travel y vistas seguras.
- Desarrollar extracción histórica e incremental, transformación, carga y calidad.
- Gestionar historial SCD Tipo 2 y agregados diarios; automatizar a las 02:00.

## 3. Tecnologías y metodología
PostgreSQL, Snowflake, Python 3.12, pandas, psycopg2, snowflake-connector-python, schedule y python-dotenv.

**Ejecutar por secciones, no Run All.** Las celdas de escritura y automatización requieren revisión manual. Este notebook es una guía de implementación, no evidencia de una ejecución realizada. Conservar los archivos oficiales sin cambios; trabajar sobre copias. No regenerar datos de PostgreSQL.

In [1]:
%pip install snowflake-connector-python schedule python-dotenv holidays

   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   --------------------- ------------------ 6.3/11.7 MB 48.2 MB/s eta 0:00:01
   ---------------------------------------- 11.7/11.7 MB 30.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 46.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 37.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os
from getpass import getpass
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path.cwd()
BASE = ROOT / 'FleetLogix'
assert BASE.is_dir(), 'Abre el notebook desde la carpeta del proyecto.'
load_dotenv(ROOT / '.env')
SQL_OFICIAL = BASE / 'A3-04_dimensional_model.sql'
ETL_OFICIAL = BASE / 'A3-05_etl_pipeline_estudiantes.py'
for archivo in (SQL_OFICIAL, ETL_OFICIAL):
    print(archivo.name, archivo.exists())
    assert archivo.exists()

A3-04_dimensional_model.sql True
A3-05_etl_pipeline_estudiantes.py True


## 4. Exploración de PostgreSQL
Solo lectura. Confirmar estructura, conteos y cobertura temporal. Introducir la contraseña en el campo oculto; no guardarla en el notebook.

In [3]:
pg = psycopg2.connect(host=os.getenv('PGHOST','localhost'), port=os.getenv('PGPORT','5432'), dbname=os.getenv('PGDATABASE','fleetlogix'), user=os.getenv('PGUSER','postgres'), password=os.getenv('PGPASSWORD') or getpass('Contraseña PostgreSQL: '))
pg.set_session(readonly=True, autocommit=True)
def consultar_pg(sql, params=None):
    with pg.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
display(consultar_pg('SELECT current_database() AS base'))

,base
0,fleetlogix


In [4]:
tablas = ['vehicles','drivers','routes','trips','deliveries','maintenance']
esquema = consultar_pg("""SELECT table_name, column_name, data_type, is_nullable
FROM information_schema.columns WHERE table_schema='public'
AND table_name=ANY(%s) ORDER BY table_name, ordinal_position""", (tablas,))
display(esquema)
conteos = pd.DataFrame([{'tabla': t, 'registros': consultar_pg(f'SELECT COUNT(*) AS n FROM public.{t}').iloc[0,0]} for t in tablas])
display(conteos)

,table_name,column_name,data_type,is_nullable
0,deliveries,delivery_id,integer,NO
1,deliveries,trip_id,integer,YES
2,deliveries,tracking_number,character varying,NO
3,deliveries,customer_name,character varying,NO
4,deliveries,delivery_address,text,NO
5,deliveries,package_weight_kg,numeric,YES
6,deliveries,scheduled_datetime,timestamp without time zone,YES
7,deliveries,delivered_datetime,timestamp without time zone,YES
8,deliveries,delivery_status,character varying,YES
9,deliveries,recipient_signature,boolean,YES


,tabla,registros
0,vehicles,200
1,drivers,400
2,routes,50
3,trips,100000
4,deliveries,400000
5,maintenance,5000


In [5]:
display(consultar_pg("""SELECT delivery_status, COUNT(*) AS registros,
MIN(delivered_datetime) AS primera, MAX(delivered_datetime) AS ultima
FROM public.deliveries GROUP BY delivery_status ORDER BY delivery_status"""))

,delivery_status,registros,primera,ultima
0,delivered,399712,2024-11-05 02:15:21.498618,2026-11-05 12:21:05.992641
1,pending,288,NaT,NaT


## 5. Modelo dimensional
Grano: una fila por entrega completada (`delivery_id`). Las seis dimensiones son fecha, tiempo, vehículo, conductor, ruta y cliente. La fecha de hecho se toma de la programación, como en el ETL oficial; la extracción incremental se basa en fecha efectiva de entrega. El tiempo usa HHMM.

| Dimensión | Origen | Clave / tratamiento |
|---|---|---|
| dim_date | Calendario colombiano | YYYYMMDD; festivos Colombia |
| dim_time | 1440 minutos del día | HHMM |
| dim_vehicle | vehicles, maintenance | Clave sustituta; SCD2 |
| dim_driver | drivers | Clave sustituta; SCD2 |
| dim_route | routes | route_id como clave inicial |
| dim_customer | deliveries y routes | Identidad aproximada: nombre y ciudad |
| fact_deliveries | deliveries, trips, routes | delivery_id único lógico |

No existe catálogo de clientes ni zona del conductor en la estructura suministrada. No inventaremos estos atributos. El historial anterior a la primera captura SCD2 no se puede reconstruir; la carga inicial usa el estado conocido como aproximación. Distancia de ruta y combustible del viaje se prorratean entre entregas del viaje para evitar doble conteo. Ingresos y costos son estimaciones en COP, según fórmulas oficiales.

In [6]:
# Inspección del SQL oficial (no ejecuta sentencias)
print(SQL_OFICIAL.read_text(encoding='utf-8'))

-- =====================================================
-- FLEETLOGIX - DATA WAREHOUSE DIMENSIONAL MODEL
-- Modelo estrella para análisis en Snowflake
-- =====================================================

-- Crear warehouse y database en Snowflake
USE ROLE ACCOUNTADMIN;
CREATE WAREHOUSE IF NOT EXISTS FLEETLOGIX_WH WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60;
CREATE DATABASE IF NOT EXISTS FLEETLOGIX_DW;
USE DATABASE FLEETLOGIX_DW;
CREATE SCHEMA IF NOT EXISTS ANALYTICS;
USE SCHEMA ANALYTICS;

-- =====================================================
-- DIMENSIONES
-- =====================================================

-- Dimensión Fecha
CREATE OR REPLACE TABLE dim_date (
    date_key INT PRIMARY KEY,
    full_date DATE NOT NULL,
    day_of_week INT,
    day_name VARCHAR(10),
    day_of_month INT,
    day_of_year INT,
    week_of_year INT,
    month_num INT,
    month_name VARCHAR(10),
    quarter INT,
    year INT,
    is_weekend BOOLEAN,
    is_holiday BOOLEAN,
    holiday_n

## 6. Configuración de Snowflake
Crear una cuenta de prueba y elegir una edición compatible con Time Travel de 30 días y warehouses multiclúster (Enterprise o superior). Confirmar edición y créditos antes de habilitarlos. XSMALL, auto-suspend 60 s y auto-resume; escalabilidad propuesta 1–2 clústeres si la edición lo admite. No ejecutar el SQL oficial repetidamente: contiene CREATE OR REPLACE TABLE, que puede reemplazar datos.

Para iniciar sesión usar las variables `SF_ACCOUNT`, `SF_USER`, `SF_ROLE`, `SF_AUTHENTICATOR` o introducir los datos en los campos solicitados. No subir `.env` a GitHub. Revisar el SQL en Snowsight y ejecutar solo después de confirmar que es una instalación nueva.

In [ ]:
import snowflake.connector
# Ejecutar únicamente después de crear la cuenta.
sf = snowflake.connector.connect(account=os.getenv('SF_ACCOUNT') or input('Identificador de cuenta: '), user=os.getenv('SF_USER') or input('Usuario Snowflake: '), authenticator=os.getenv('SF_AUTHENTICATOR','externalbrowser'), role=os.getenv('SF_ROLE','ACCOUNTADMIN'))
def consultar_sf(sql, params=None):
    with sf.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
display(consultar_sf('SELECT CURRENT_ACCOUNT(), CURRENT_ROLE()'))

In [ ]:
# Generar borrador no destructivo para revisar en Snowsight, sin ejecutarlo.
modelo = SQL_OFICIAL.read_text(encoding='utf-8')
modelo = modelo.replace('CREATE OR REPLACE TABLE', 'CREATE TABLE IF NOT EXISTS')
modelo = modelo.replace('CREATE OR REPLACE SECURE VIEW', 'CREATE SECURE VIEW IF NOT EXISTS')
print(modelo)
print('\n-- Propuesta adicional (solo edición compatible):')
print('ALTER WAREHOUSE FLEETLOGIX_WH SET MIN_CLUSTER_COUNT=1 MAX_CLUSTER_COUNT=2 AUTO_RESUME=TRUE;')

### 6.1 Seguridad y verificación
El filtro oficial excluye clientes Gobierno; no garantiza que cada vendedor vea solo sus clientes. Definir asignaciones reales por usuario/cliente antes de otorgar acceso. Los roles analíticos requieren USAGE en warehouse, base y esquema, y SELECT únicamente en sus vistas. Operaciones puede recibir acceso de lectura a los hechos y dimensiones si se confirma el requerimiento. Probar con usuarios sin roles administrativos. No conceder acceso a ventas hasta validar el aislamiento.

In [ ]:
# Ejecutar después de crear el modelo manualmente.
display(consultar_sf('SHOW TABLES IN SCHEMA FLEETLOGIX_DW.ANALYTICS'))
display(consultar_sf('SHOW VIEWS IN SCHEMA FLEETLOGIX_DW.ANALYTICS'))
display(consultar_sf("SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE FLEETLOGIX_DW.ANALYTICS.FACT_DELIVERIES"))

## 7. Extracción — completar el primer TO DO
Se extraen entregas completadas con su viaje y ruta. Se cuentan todas las entregas del viaje antes de aplicar el filtro temporal. La carga histórica no usa filtro; la diaria usa un intervalo semiabierto. La ausencia de updated_at limita la detección de correcciones tardías: se propone ventana de reproceso y MERGE por delivery_id.

In [ ]:
SQL_EXTRACT = """
WITH cantidades AS (
 SELECT trip_id, COUNT(*) AS deliveries_in_trip,
 COUNT(*) FILTER (WHERE delivery_status='delivered' AND delivered_datetime IS NOT NULL) AS completed_in_trip
 FROM public.deliveries GROUP BY trip_id
)
SELECT d.*, t.vehicle_id, t.driver_id, t.route_id,
 t.departure_datetime, t.arrival_datetime, t.fuel_consumed_liters,
 r.distance_km, r.toll_cost, r.destination_city,
 q.deliveries_in_trip, q.completed_in_trip
FROM public.deliveries d
LEFT JOIN public.trips t ON d.trip_id=t.trip_id
LEFT JOIN public.routes r ON t.route_id=r.route_id
LEFT JOIN cantidades q ON d.trip_id=q.trip_id
WHERE d.delivery_status='delivered'
"""
def extraer(inicio=None, fin=None, limite=None):
    sql, params = SQL_EXTRACT, []
    if inicio is not None and fin is not None:
        sql += ' AND d.delivered_datetime >= %s AND d.delivered_datetime < %s'
        params.extend([inicio, fin])
    sql += ' ORDER BY d.delivery_id'
    if limite is not None:
        sql += ' LIMIT %s'; params.append(limite)
    return consultar_pg(sql, tuple(params))
muestra = extraer(limite=1000)
print('Registros de prueba:',len(muestra)); display(muestra.head())

## 8. Transformación y calidad
Duración real: entrega menos salida del viaje. Retraso: máximo entre cero y entrega menos programación. Una entrega anticipada puede ser válida. Puntualidad: tolerancia de 30 minutos. La tasa entregas/hora es por viaje y no debe sumarse entre entregas; para desempeño por conductor se agregan los viajes una sola vez. Duraciones negativas se rechazan y documentan. No modificar el origen.

In [ ]:
def transformar(df):
    x = df.copy()
    for col in ['departure_datetime','arrival_datetime','scheduled_datetime','delivered_datetime']:
        x[col] = pd.to_datetime(x[col], errors='coerce')
    for col in ['distance_km','fuel_consumed_liters','toll_cost','package_weight_kg','deliveries_in_trip','completed_in_trip']:
        x[col] = pd.to_numeric(x[col], errors='coerce')
    dur = (x.delivered_datetime-x.departure_datetime).dt.total_seconds()/60
    retraso = (x.delivered_datetime-x.scheduled_datetime).dt.total_seconds()/60
    horas = (x.arrival_datetime-x.departure_datetime).dt.total_seconds()/3600
    valido = (dur.ge(0) & horas.gt(0) & retraso.notna() & x.package_weight_kg.between(0,10000,inclusive='neither') & x.deliveries_in_trip.gt(0) & x[['vehicle_id','driver_id','route_id','customer_name','destination_city']].notna().all(axis=1) & x.distance_km.ge(0) & x.fuel_consumed_liters.ge(0) & x.toll_cost.ge(0) & ~x.delivery_id.duplicated(keep=False))
    rechazados = x.loc[~valido].copy()
    x['delivery_time_minutes'] = dur.round()
    x['delay_minutes'] = retraso.clip(lower=0).round()
    x['is_on_time'] = retraso.le(30)
    x['deliveries_per_hour'] = x.completed_in_trip / horas.where(horas.gt(0))
    x['fuel_efficiency_km_per_liter'] = x.distance_km / x.fuel_consumed_liters.where(x.fuel_consumed_liters.gt(0))
    x['cost_per_delivery'] = (x.fuel_consumed_liters*5000+x.toll_cost)/x.deliveries_in_trip
    x['revenue_per_delivery'] = 20000+x.package_weight_kg*500
    x['distance_km'] /= x.deliveries_in_trip
    x['fuel_consumed_liters'] /= x.deliveries_in_trip
    x['date_key'] = x.scheduled_datetime.dt.strftime('%Y%m%d').astype('Int64')
    for origen, destino in [('scheduled_datetime','scheduled_time_key'),('delivered_datetime','delivered_time_key')]:
        x[destino] = (x[origen].dt.hour*100+x[origen].dt.minute).astype('Int64')
    x['has_signature'] = x.recipient_signature
    x['is_damaged'] = None # No existe dato de daño confirmado en el origen
    return x.loc[valido].copy(), rechazados
transformadas, rechazadas = transformar(muestra)
print('Extraídas:',len(muestra),'Válidas:',len(transformadas),'Rechazadas:',len(rechazadas))
assert len(muestra)==len(transformadas)+len(rechazadas)
assert transformadas.delivery_id.is_unique
display(transformadas.head())

## 9. Dimensiones: preparación local
Las siguientes celdas preparan calendarios y catálogos; no escriben en Snowflake. Las reglas de negocio no especificadas (turnos, categorías) son propuestas a revisar. Los festivos corresponden a Colombia; trimestre fiscal se iguala al calendario como supuesto.

In [ ]:
import holidays
rango = consultar_pg("SELECT MIN(scheduled_datetime) AS inicio, MAX(scheduled_datetime) AS fin FROM public.deliveries").iloc[0]
fechas = pd.date_range(pd.Timestamp(rango.inicio).normalize(), max(pd.Timestamp(rango.fin).normalize(),pd.Timestamp.today().normalize())+pd.Timedelta(days=366))
festivos = holidays.country_holidays('CO', years=sorted(set(fechas.year)), language='es')
dim_date = pd.DataFrame({'full_date':fechas.date,'date_key':fechas.strftime('%Y%m%d').astype(int),'day_of_week':fechas.dayofweek+1,'day_of_month':fechas.day,'day_of_year':fechas.dayofyear,'week_of_year':fechas.isocalendar().week.to_numpy(),'month_num':fechas.month,'quarter':fechas.quarter,'year':fechas.year,'is_weekend':fechas.dayofweek>=5})
dim_date['day_name'] = [ ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo'][i] for i in fechas.dayofweek ]
dim_date['month_name'] = [ ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre'][i-1] for i in fechas.month ]
dim_date['holiday_name'] = [festivos.get(d) for d in fechas.date]
dim_date['is_holiday'] = dim_date.holiday_name.notna()
dim_date['fiscal_quarter'] = dim_date.quarter; dim_date['fiscal_year'] = dim_date.year
display(dim_date.head())

In [ ]:
minutos = pd.date_range('2000-01-01', periods=1440, freq='min')
h = minutos.hour
dim_time = pd.DataFrame({'time_key':h*100+minutos.minute,'hour':h,'minute':minutos.minute,'second':0,'hour_24':minutos.strftime('%H:%M'),'hour_12':minutos.strftime('%I:%M %p'),'am_pm':minutos.strftime('%p'),'is_business_hour':(h>=8)&(h<18),'shift':np.select([h<8,h<16],['Turno 1','Turno 2'],default='Turno 3'),'time_of_day':np.select([h<6,h<12,h<18],['Madrugada','Mañana','Tarde'],default='Noche')})
display(dim_time.head()); assert len(dim_time)==1440

In [ ]:
dim_route = consultar_pg('SELECT * FROM public.routes')
dim_route['route_key'] = dim_route.route_id
# Sin datos oficiales de dificultad ni tipo: conservar desconocidos.
dim_route['difficulty_level'] = None; dim_route['route_type'] = None
vehiculos = consultar_pg("""SELECT v.*, (SELECT MAX(m.maintenance_date) FROM public.maintenance m WHERE m.vehicle_id=v.vehicle_id) AS last_maintenance_date FROM public.vehicles v""")
conductores = consultar_pg("SELECT *, concat_ws(' ', first_name,last_name) AS full_name FROM public.drivers")
clientes = consultar_pg("""SELECT d.customer_name,r.destination_city AS city, MIN(d.delivered_datetime)::date AS first_delivery_date, COUNT(*) AS total_deliveries FROM public.deliveries d JOIN public.trips t USING(trip_id) JOIN public.routes r USING(route_id) WHERE d.delivery_status='delivered' AND d.delivered_datetime IS NOT NULL GROUP BY 1,2""")
for nombre, df in [('rutas',dim_route),('vehículos',vehiculos),('conductores',conductores),('clientes',clientes)]:
    print(nombre,len(df))

## 10. Carga dimensional y SCD Tipo 2 — implementación y evidencia
La implementación ejecutable de carga de las seis dimensiones y gestión histórica SCD Tipo 2 está en `A3-05_etl_pipeline_FleetLogix.ipynb`, clase `FleetLogixETL`. El notebook principal conserva la exploración y preparación del modelo; el pipeline separado realiza la carga mediante staging y operaciones MERGE.

**Evidencia:** se completó la carga histórica y una carga incremental posterior. La validación detallada de cambios SCD2 requiere consultar las versiones de cada dimensión; no se infiere únicamente del conteo de hechos.


### Referencia de ejecución
Abrir `A3-05_etl_pipeline_FleetLogix.ipynb` y ejecutar la clase `FleetLogixETL` con sus credenciales externas. **No ejecutar `Run All` en este notebook principal**: contiene celdas de exploración que solicitan credenciales y ejemplos de preparación. Las pruebas ya realizadas se documentan en el notebook ETL.


In [ ]:
# Verificación de dimensiones (después de la carga dimensional)
for tabla, clave in [('DIM_DATE','DATE_KEY'),('DIM_TIME','TIME_KEY'),('DIM_VEHICLE','VEHICLE_KEY'),('DIM_DRIVER','DRIVER_KEY'),('DIM_ROUTE','ROUTE_KEY'),('DIM_CUSTOMER','CUSTOMER_KEY')]:
    display(consultar_sf(f'SELECT COUNT(*) AS TOTAL, COUNT(DISTINCT {clave}) AS CLAVES FROM FLEETLOGIX_DW.ANALYTICS.{tabla}'))

## 11. Carga de hechos — implementada en el pipeline ETL
La clase `FleetLogixETL` carga `FACT_DELIVERIES` mediante staging y `MERGE` por `delivery_id`, evitando insertar dos veces la misma entrega. Se ejecutó una carga histórica de **399,712** hechos y posteriormente una carga incremental de la fecha **2026-11-04**.

**Resultado incremental:** 548 filas extraídas, 548 transformadas, 0 rechazadas y 399,716 hechos al finalizar. La consulta de Snowflake confirmó **4 entregas nuevas** (`399569`, `399570`, `399571`, `399572`) y **0 duplicados**. Las 548 filas corresponden al lote de esa fecha, no a 548 hechos nuevos.


## 12. Agregados diarios — segundo y tercer TO DO
Propuesta de SQL para revisar. Los totales se recalculan desde los hechos para que el reproceso no duplique resultados. Los ingresos y costos son estimados; la distancia y combustible son asignaciones por entrega.

In [ ]:
SQL_CREAR_TOTALES = """CREATE TABLE IF NOT EXISTS daily_delivery_totals (
 date_key INT PRIMARY KEY, total_deliveries INT, on_time_deliveries INT,
 total_weight_kg NUMBER(18,2), total_revenue NUMBER(18,2), total_cost NUMBER(18,2))"""
SQL_MERGE_TOTALES = """MERGE INTO daily_delivery_totals t USING (
 SELECT date_key, COUNT(*) AS total_deliveries, COUNT_IF(is_on_time) AS on_time_deliveries,
 SUM(package_weight_kg) AS total_weight_kg, SUM(revenue_per_delivery) AS total_revenue,
 SUM(cost_per_delivery) AS total_cost FROM fact_deliveries GROUP BY date_key
) s ON t.date_key=s.date_key
WHEN MATCHED THEN UPDATE SET t.total_deliveries=s.total_deliveries,t.on_time_deliveries=s.on_time_deliveries,t.total_weight_kg=s.total_weight_kg,t.total_revenue=s.total_revenue,t.total_cost=s.total_cost
WHEN NOT MATCHED THEN INSERT (date_key,total_deliveries,on_time_deliveries,total_weight_kg,total_revenue,total_cost)
VALUES(s.date_key,s.total_deliveries,s.on_time_deliveries,s.total_weight_kg,s.total_revenue,s.total_cost)"""
print(SQL_CREAR_TOTALES); print(SQL_MERGE_TOTALES)

## 13. Automatización nocturna — prueba controlada
El pipeline incluye una configuración opcional con `schedule` a las **02:00**, zona `America/Bogota`. El 16/11/2026 se ejecutó una prueba programada cada 10 segundos; el proceso se autenticó y finalizó sin errores, sin encontrar entregas nuevas para el periodo automático.

**Alcance:** la programación permanente está desactivada (`ACTIVAR_AUTOMATIZACION=False`) y requiere un proceso activo o un programador del sistema operativo. La prueba incremental posterior con cuatro nuevas entregas se ejecutó manualmente, no por la programación nocturna.


In [ ]:
# Referencia a la implementación real: A3-05_etl_pipeline_FleetLogix.ipynb
# La automatización se encuentra desactivada por defecto para evitar procesos persistentes en Jupyter.
print("Automatización documentada; activación permanente pendiente de despliegue.")


## 14. Validaciones y evidencia final
Ejecutar después de implementar las cargas y comprobar permisos. Las PK/FK de tablas estándar Snowflake no sustituyen estas comprobaciones. Validar también reconciliación origen-destino, rechazados, repetición de lote y pruebas SCD2 y de roles.

In [ ]:
display(consultar_sf("""SELECT COUNT(*) AS filas, COUNT(DISTINCT delivery_id) AS entregas,
 COUNT_IF(delivery_time_minutes<0) AS tiempos_negativos, COUNT_IF(delivery_status IS NULL OR delivery_status<>'delivered') AS estados_invalidos
 FROM FLEETLOGIX_DW.ANALYTICS.fact_deliveries"""))
for tabla, fk, pk in [('dim_date','date_key','date_key'),('dim_time','scheduled_time_key','time_key'),('dim_time','delivered_time_key','time_key'),('dim_vehicle','vehicle_key','vehicle_key'),('dim_driver','driver_key','driver_key'),('dim_route','route_key','route_key'),('dim_customer','customer_key','customer_key')]:
    display(consultar_sf(f'SELECT COUNT(*) AS huerfanos FROM FLEETLOGIX_DW.ANALYTICS.fact_deliveries f LEFT JOIN FLEETLOGIX_DW.ANALYTICS.{tabla} d ON f.{fk}=d.{pk} WHERE d.{pk} IS NULL'))

## 15. Evidencias y conclusiones
| Evidencia | Estado verificado |
|---|---|
| PostgreSQL | 400,000 entregas en el conjunto original; 399,712 completadas antes de la simulación incremental |
| Modelo dimensional | Seis dimensiones y tabla de hechos implementadas en Snowflake; ver pipeline y SQL del modelo |
| ETL histórico | 399,712 hechos cargados; 0 rechazados en la ejecución reportada |
| ETL incremental | 548 filas del día procesadas; 4 entregas nuevas; total 399,716 hechos |
| Duplicados en hechos | 0 según consulta posterior a la carga incremental |
| Vistas seguras | `V_OPERATIONS_DELIVERIES` y `V_SALES_DELIVERIES` verificadas con `IS_SECURE=YES` |
| Time Travel | Retención `DATA_RETENTION_TIME_IN_DAYS=30` verificada en `FACT_DELIVERIES` |
| Automatización | Prueba de `schedule` satisfactoria; ejecución permanente no desplegada |
| SCD2 | Lógica implementada; revisión exhaustiva de versiones históricas no documentada aquí |

### Conclusión
Se construyó y probó un Data Warehouse analítico en Snowflake alimentado desde PostgreSQL. El proceso ETL demostró carga histórica y actualización incremental con cuatro entregas nuevas sin duplicados, y se comprobaron vistas seguras y retención de Time Travel. Para un entorno productivo aún se requiere desplegar la programación permanente, gestionar credenciales mediante un almacén seguro y ampliar las pruebas de cambios SCD2.


In [ ]:
# Ejecutar al finalizar la sesión.
for conexion in ('pg','sf'):
    if conexion in globals():
        globals()[conexion].close()
        print(conexion, 'cerrada')